# 01. Case Base — Data Acquisition & Preprocessing

Tahap ini digunakan untuk membangun **case base** putusan perkara **perdata waris**.

Sesuai instruksi tugas, tahap ini mencakup:

1. Menyiapkan struktur folder project.
2. Mengambil minimal 30 dokumen putusan dalam format PDF.
3. Mengonversi PDF menjadi teks.
4. Membersihkan teks dari header, footer, watermark, nomor halaman, dan karakter tidak penting.
5. Menyimpan hasil teks bersih ke folder `data/raw/`.
6. Membuat log validasi pembersihan di folder `logs/`.

> Letakkan file PDF putusan asli di folder `data/pdf/`, bukan di `data/raw/`.


## 1. Instalasi dan Import Library

In [1]:
%pip install pypdf pandas tqdm -q

import os
import re
import shutil
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from pypdf import PdfReader

print("Library siap digunakan.")


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Library siap digunakan.


## 2. Membuat Struktur Folder

Struktur folder mengikuti instruksi tugas:

```text
/data/
  /raw/        -> hasil ekstraksi teks bersih
  /processed/  -> hasil representasi kasus tahap 2
  /eval/       -> query dan evaluasi
  /results/    -> hasil prediksi
/logs/         -> cleaning.log dan cleaning.csv
```

In [4]:
# Gunakan path relatif dari notebook.
# Jika notebook berada di folder /notebooks/, maka root project adalah satu folder di atasnya.
NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_TXT_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EVAL_DIR = DATA_DIR / "eval"
RESULTS_DIR = DATA_DIR / "results"
LOG_DIR = PROJECT_ROOT / "logs"

FOLDERS = [
    RAW_TXT_DIR,
    PROCESSED_DIR,
    EVAL_DIR,
    RESULTS_DIR,
    LOG_DIR,
]

for folder in FOLDERS:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("RAW TXT folder:", RAW_TXT_DIR)
print("LOG folder   :", LOG_DIR)

Project root : d:\Semester 6\Penalaran Komputer\Tugas 3 -PK
RAW TXT folder: d:\Semester 6\Penalaran Komputer\Tugas 3 -PK\data\raw
LOG folder   : d:\Semester 6\Penalaran Komputer\Tugas 3 -PK\logs


## 3. Fungsi Ekstraksi Teks dari PDF

Fungsi ini membaca setiap halaman PDF dan menggabungkan hasil ekstraksi menjadi satu teks utuh.


In [17]:
import os
from pypdf import PdfReader

# Tentukan path folder data raw
raw_data_dir = "../data/raw/"

# Ambil semua file yang berakhiran .pdf di dalam folder tersebut
pdf_files = [f for f in os.listdir(raw_data_dir) if f.endswith('.pdf')]

print(f"Menemukan {len(pdf_files)} file PDF untuk dikonversi.")

# Proses konversi setiap file PDF
for index, pdf_file in enumerate(pdf_files, start=1):
    pdf_path = os.path.join(raw_data_dir, pdf_file)
    
    # Tentukan nama file .txt keluaran (misal: case_001.txt)
    txt_filename = f"case_{index:03d}.txt"
    txt_path = os.path.join(raw_data_dir, txt_filename)
    
    try:
        # Membaca file PDF
        reader = PdfReader(pdf_path)
        extracted_text = []
        
        # Ekstrak teks dari setiap halaman PDF
        for page in reader.pages:
            text = page.extract_text()
            if text:
                extracted_text.append(text)
        
        # Gabungkan teks seluruh halaman menjadi satu string
        full_text = "\n".join(extracted_text)
        
        # Simpan hasil ekstraksi teks ke file .txt
        with open(txt_path, "w", encoding="utf-8") as txt_file:
            txt_file.write(full_text)
            
        print(f"✅ Berhasil mengonversi: {pdf_file} -> {txt_filename}")
        
    except Exception as e:
        print(f"❌ Gagal memproses {pdf_file}. Error: {e}")

print("\nProses konversi selesai!")

Menemukan 50 file PDF untuk dikonversi.
✅ Berhasil mengonversi: case_001 .pdf -> case_001.txt
✅ Berhasil mengonversi: case_002.pdf -> case_002.txt
✅ Berhasil mengonversi: case_003.pdf -> case_003.txt
✅ Berhasil mengonversi: case_004.pdf -> case_004.txt
✅ Berhasil mengonversi: case_005.pdf -> case_005.txt
✅ Berhasil mengonversi: case_006.pdf -> case_006.txt
✅ Berhasil mengonversi: case_007.pdf -> case_007.txt
✅ Berhasil mengonversi: case_008.pdf -> case_008.txt
✅ Berhasil mengonversi: case_009.pdf -> case_009.txt
✅ Berhasil mengonversi: case_010.pdf -> case_010.txt
✅ Berhasil mengonversi: case_011.pdf -> case_011.txt
✅ Berhasil mengonversi: case_012.pdf -> case_012.txt
✅ Berhasil mengonversi: case_013.pdf -> case_013.txt
✅ Berhasil mengonversi: case_014.pdf -> case_014.txt
✅ Berhasil mengonversi: case_015.pdf -> case_015.txt
✅ Berhasil mengonversi: case_016.pdf -> case_016.txt
✅ Berhasil mengonversi: case_017.pdf -> case_017.txt
✅ Berhasil mengonversi: case_018.pdf -> case_018.txt
✅ Ber

## 4. Fungsi Pembersihan Teks

Pembersihan dibuat **tidak terlalu agresif** agar informasi hukum tetap aman, misalnya:

- nomor perkara,
- tanggal,
- pasal,
- nama pihak,
- amar putusan,
- tanda `/`, `.`, `,`, `:`, `;`, `-`, dan tanda kurung.

Yang dihapus hanya bagian yang sering menjadi noise PDF Mahkamah Agung, seperti header, footer, watermark, nomor halaman, dan disclaimer.


In [27]:
import os
import re
import pandas as pd  # Ditambahkan untuk handle CSV

# Tentukan path folder data raw
raw_data_dir = "../data/raw/"
path_log_utama = "../logs/cleaning.log"
path_csv_utama = "../logs/cleaning.csv"  # Path untuk CSV baru

# Ambil semua file .txt hasil konversi awal
txt_files = [f for f in os.listdir(raw_data_dir) if f.endswith('.txt') and f.startswith('case_')]
txt_files.sort()

print(f"Menemukan {len(txt_files)} file teks untuk dibersihkan.\n")

# Penampung untuk log TXT dan CSV
log_lines = ["=== LOG FILE PEMBERSIHAN DATA PUTUSAN ==="]
csv_data_list = []  # List untuk menampung baris data CSV

for txt_file in txt_files:
    file_path = os.path.join(raw_data_dir, txt_file)
    
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()
    
    len_original = len(text)
    
    # --- PROSES PEMBERSIHAN YANG AMAN ---
    cleaned_text = text.lower()
    
    pola_sampah = [
        r"direktori\s+putusan\s+mahkamah\s+agung\s+republik\s+indonesia",
        r"putusan\.mahkamahagung\.go\.id",
        r"mahkamah\s+agung\s+republik\s+indonesia",
        r"kepaniteraan\s+mahkamah\s+agung.*",
        r"direktorat\s+jenderal\s+badan\s+peradilan.*",
        r"halaman\s+\d+\s+dari\s+\d+.*",
        r"hal\s+\d+\s+put\s+no.*",
        r"page\s+\d+",
        r"disclaimer.*",
        r"dalam\s+hal\s+anda\s+menemukan\s+incompabilitas.*",
        r"dalam\s+hal\s+anda\s+menemukan\s+inkompatibilitas.*",
    ]
    for pola in pola_sampah:
        cleaned_text = re.sub(pola, "", cleaned_text)
    
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    cleaned_text = re.sub(r'[^a-z0-9\s\/\(\)\,\.\-]', '', cleaned_text)
    
    len_cleaned = len(cleaned_text)
    ratio = (len_cleaned / len_original) * 100 if len_original > 0 else 0
    
    # Biar sinkron dengan kode validasi kamu, status dibuat ringkas: "VALID" / "PERLU CEK"
    status_validasi = "VALID" if ratio >= 80 else "PERLU CEK"
    status_log = f"VALID (>= 80% isi tersedia)" if ratio >= 80 else "WARNING (terlalu banyak terhapus)"
    
    # Simpan kembali teks yang sudah bersih
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(cleaned_text)
    
    # 1. Masukkan data ke log TXT
    log_info = f"File: {txt_file}\n - Karakter Asli: {len_original}\n - Karakter Bersih: {len_cleaned}\n - Rasio Keutuhan: {ratio:.2f}%\n - Status: {status_log}\n" + "-"*40
    print(f"✅ Selesai memproses: {txt_file} ({ratio:.2f}% data dipertahankan)")
    log_lines.append(log_info)
    
    # 2. Masukkan data ke struktur CSV
    csv_data_list.append({
        "case_id": txt_file.replace(".txt", ""),
        "txt_file": txt_file,
        "raw_words": len_original,        # Menggunakan jumlah karakter sebagai representasi
        "cleaned_words": len_cleaned,
        "coverage_percent": round(ratio, 2),
        "status": status_validasi
    })

# --- SIMPAN LOG TXT ---
with open(path_log_utama, "a", encoding="utf-8") as log_file:
    log_file.write("\n\n=== RERUN PREPROCESSING: ADAPTIF PASAL & PIHAK ===\n")
    log_file.write("\n".join(log_lines))

# --- SIMPAN ATAU MERGE LOG CSV ---
new_df = pd.DataFrame(csv_data_list)

if os.path.exists(path_csv_utama) and os.path.getsize(path_csv_utama) > 0:
    # Jika file csv lama sudah ada isinya, kita gabung (append) datanya
    old_df = pd.read_csv(path_csv_utama)
    combined_df = pd.concat([old_df, new_df], ignore_index=True).drop_duplicates(subset=['case_id'], keep='last')
    combined_df.to_csv(path_csv_utama, index=False)
else:
    # Jika belum ada atau masih kosong, langsung buat baru
    new_df.to_csv(path_csv_utama, index=False)

print(f"\nProses Preprocessing Ulang Selesai!")
print(f"Log TXT digabung ke : {path_log_utama}")
print(f"Log CSV disimpan ke  : {path_csv_utama}")

Menemukan 50 file teks untuk dibersihkan.

✅ Selesai memproses: case_001.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_002.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_003.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_004.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_005.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_006.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_007.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_008.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_009.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_010.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_011.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_012.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_013.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_014.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_015.txt (100.00% data dipertahankan)
✅ Selesai me

## 5. Fungsi Validasi Keutuhan Teks

Coverage dihitung berdasarkan jumlah kata:

```text
coverage = jumlah kata setelah dibersihkan / jumlah kata hasil ekstraksi PDF × 100
```

Target validasi dibuat mengikuti instruksi tugas, yaitu minimal **80% isi putusan tersedia**.

Catatan: jika coverage sedikit di bawah 80%, cek manual isi file `.txt`. Bisa jadi PDF memiliki banyak header/footer/disclaimer yang memang harus dibuang.


In [28]:
def calculate_coverage(raw_text: str, cleaned_text: str) -> tuple:
    raw_words = len(raw_text.split())
    cleaned_words = len(cleaned_text.split())

    if raw_words == 0:
        coverage = 0.0
    else:
        coverage = round((cleaned_words / raw_words) * 100, 2)

    return raw_words, cleaned_words, coverage


def check_key_sections(text: str) -> dict:
    """Cek sederhana apakah bagian penting putusan masih tersedia."""
    keywords = {
        "nomor_perkara": bool(re.search(r"nomor|no\.", text, flags=re.IGNORECASE)),
        "menimbang": "menimbang" in text,
        "mengingat": "mengingat" in text,
        "mengadili": "mengadili" in text,
        "pasal": "pasal" in text,
        "putusan": "putusan" in text,
    }

    score = round((sum(keywords.values()) / len(keywords)) * 100, 2)
    keywords["section_score_percent"] = score

    return keywords


print("Fungsi validasi siap digunakan.")

Fungsi validasi siap digunakan.


## 6. Proses Konversi PDF ke TXT dan Pembersihan

Output utama tahap ini adalah:

```text
data/raw/case_001.txt
data/raw/case_002.txt
...
logs/cleaning.log
logs/cleaning.csv
```


In [29]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm

# Setup folder data
PDF_DIR = Path("raw")
LOG_DIR = Path("logs")
RAW_TXT_DIR = Path("extracted_text")

# Bikin folder otomatis kalau belum ada
LOG_DIR.mkdir(parents=True, exist_ok=True)
RAW_TXT_DIR.mkdir(parents=True, exist_ok=True)

# Setup path file log
LOG_TXT_PATH = LOG_DIR / "cleaning.log"
LOG_CSV_PATH = LOG_DIR / "cleaning.csv"

# Ambil semua file pdf di folder raw
pdf_files = sorted(PDF_DIR.glob("*.pdf"))
cleaning_log = []

print(f"Mulai memproses {len(pdf_files)} file PDF...\n")

for index, pdf_path in enumerate(tqdm(pdf_files, desc="Case Base"), start=1):
    case_id = f"case_{index:03d}"
    txt_path = RAW_TXT_DIR / f"{case_id}.txt"

    # Proses ekstraksi dan pembersihan text
    raw_text = extract_text_from_pdf(pdf_path)
    cleaned_text = clean_legal_text(raw_text)

    # Hitung nilai coverage dan cek section penting
    raw_words, cleaned_words, coverage = calculate_coverage(raw_text, cleaned_text)
    section_check = check_key_sections(cleaned_text)

    # Tentukan status berdasarkan nilai coverage
    status = "VALID" if coverage >= 80 else "PERLU CEK"

    # Simpan hasil pembersihan ke file txt
    with open(txt_path, "w", encoding="utf-8") as file:
        file.write(cleaned_text)

    # Susun baris log untuk dataframe
    log_row = {
        "case_id": case_id,
        "txt_file": txt_path.name,
        "raw_words": raw_words,
        "cleaned_words": cleaned_words,
        "coverage_percent": coverage,
        "status": status,
        **section_check,
    }
    cleaning_log.append(log_row)

    print(f"{status}: {pdf_path.name} -> {txt_path.name} | coverage={coverage}% | section_score={section_check['section_score_percent']}%")

# Convert log ke dataframe
log_df = pd.DataFrame(cleaning_log)

# Export ke CSV
log_df.to_csv(LOG_CSV_PATH, index=False)

# Export ke format ringkasan .txt
with open(LOG_TXT_PATH, "w", encoding="utf-8") as log_file:
    log_file.write("=== LOG FILE PEMBERSIHAN DATA PUTUSAN ===\n")
    log_file.write("Tahap 1: Case Base - Data Acquisition & Preprocessing\n\n")
    log_file.write(log_df.to_string(index=False))

print("\nOutput tahap 1 selesai.")
print(f"TXT folder  : {RAW_TXT_DIR}")
print(f"Cleaning log: {LOG_TXT_PATH}")
print(f"Cleaning CSV: {LOG_CSV_PATH}")

Mulai memproses 0 file PDF...



Case Base: 0it [00:00, ?it/s]


Output tahap 1 selesai.
TXT folder  : extracted_text
Cleaning log: logs\cleaning.log
Cleaning CSV: logs\cleaning.csv


## 7. Ringkasan Validasi

Bagian ini menampilkan jumlah dokumen valid, dokumen yang perlu dicek, dan contoh log pembersihan.


In [32]:
import os
import pandas as pd
from pathlib import Path

# Samakan path-nya dengan kode pembersihan yang kamu jalankan sebelumnya
LOG_CSV_PATH_FIX = Path("../logs/cleaning.csv")

if LOG_CSV_PATH_FIX.exists():
    try:
        # Coba baca file csv dari path yang benar
        log_df = pd.read_csv(LOG_CSV_PATH_FIX)
        
        if log_df.empty:
            print("File cleaning.csv berhasil dibaca, tetapi tidak ada data di dalamnya.")
        else:
            total_docs = len(log_df)
            valid_docs = (log_df["status"] == "VALID").sum()
            warning_docs = (log_df["status"] == "PERLU CEK").sum()

            print("=== RINGKASAN VALIDASI TAHAP 1 ===")
            print(f"Total dokumen diproses : {total_docs}")
            print(f"Dokumen valid          : {valid_docs}")
            print(f"Dokumen perlu cek      : {warning_docs}")
            print(f"Rata-rata coverage     : {log_df['coverage_percent'].mean():.2f}%")

            if total_docs >= 30:
                print("Syarat jumlah dokumen minimal 30: TERPENUHI")
            else:
                print("Syarat jumlah dokumen minimal 30: BELUM TERPENUHI")

            display(log_df.head())

    except pd.errors.EmptyDataError:
        print("Peringatan: File cleaning.csv yang dibaca masih kosong.")
else:
    print(f"File cleaning.csv tidak ditemukan di {LOG_CSV_PATH_FIX.resolve()}")

=== RINGKASAN VALIDASI TAHAP 1 ===
Total dokumen diproses : 50
Dokumen valid          : 50
Dokumen perlu cek      : 0
Rata-rata coverage     : 100.00%
Syarat jumlah dokumen minimal 30: TERPENUHI


,case_id,txt_file,raw_words,cleaned_words,coverage_percent,status
0,case_001,case_001.txt,27116,27116,100.0,VALID
1,case_002,case_002.txt,26653,26653,100.0,VALID
2,case_003,case_003.txt,24124,24124,100.0,VALID
3,case_004,case_004.txt,32064,32064,100.0,VALID
4,case_005,case_005.txt,23143,23143,100.0,VALID


## 9. Melihat Contoh Hasil Teks Bersih

Gunakan bagian ini untuk memastikan hasil `.txt` masih memuat isi penting putusan.


In [34]:
RAW_TXT_DIR = Path("../data/raw/")
txt_files = sorted(RAW_TXT_DIR.glob("case_*.txt"))

print(f"Total file TXT: {len(txt_files)}")

if txt_files:
    sample_path = txt_files[0]

    with open(sample_path, "r", encoding="utf-8") as file:
        sample_text = file.read()

    print(f"Contoh file: {sample_path.name}")
    print("=" * 80)
    print(sample_text[:2000])
else:
    print("Belum ada file TXT di data/raw/.")

Total file TXT: 50
Contoh file: case_001.txt
p u t u s a n nomor 62 pk/pdt/2025 demi keadilan berdasarkan ketuhanan yang maha esa m a h k a m a h a g u n g memeriksa perkara perdata dalam pemeriksaan peninjauan kembali telah memutus sebagai berikut dalam perkara antara 1. yohana s. nugraheni, bertempat tinggal di jalan pondok raya i, nomor 18, rt 001, rw 006, kelurahan pela mampang, mampang prapatan, kota jakarta selatan, dki jakarta 2. irma savira firdaus, s.h., bertempat tinggal di jalan raya mauk, km. 12, nomor 6, oja pln, desa pisangan jaya, sepatan, kabupaten tangerang, banten dalam hal ini keduanya memberi kuasa kepada edward michael anggarawan, s.h., dan kawan -kawan, para advokat pada firma hukum melf , ber kantor di axa tower, ceo suite 45 th floor, jalan prof. dr. satrio, kaveling 18, kelurahan kuningan, kecamatan setiabudi, kota jakarta selatan , berdasarkan surat kuasa khusus masing-masing tanggal 19 juli 2024 para pemohon peninjauan kembali l a w a n 1. audrick farel nolan

## 10. Kesimpulan Tahap 1

Tahap 1 menghasilkan case base awal berupa file teks putusan yang sudah dibersihkan.

Output yang dihasilkan:

1. Folder `data/raw/` berisi file `case_001.txt`, `case_002.txt`, dan seterusnya.
2. File `logs/cleaning.log` sebagai catatan proses pembersihan.
3. File `logs/cleaning.csv` sebagai rekap validasi coverage.
4. Data siap digunakan pada tahap berikutnya, yaitu **Case Representation**.
